In [ ]:
# Cell 0: Debug Path Information
import os

# Print current working directory
print(f"Current working directory: {os.getcwd()}")

# Print code file directory
print(f"Code file directory: {os.path.dirname(os.path.abspath('.'))}")

# Test relative paths
test_paths = [
    ('../Model/Best_Flowformer.pth', 'Model file'),
    ('../Scaler/scaler_input.pkl', 'Input scaler'),
    ('../Scaler/scaler_output.pkl', 'Output scaler'),
    ('../TestData/Condition_1_32_sametime.xlsx', 'Test data 1')
]

print("\nChecking relative paths:")
for rel_path, description in test_paths:
    abs_path = os.path.abspath(rel_path)
    exists = os.path.exists(abs_path)
    status = "✓" if exists else "✗"
    print(f"  {status} {description}: {abs_path}")
    if not exists:
        print(f"    Relative path: {rel_path}")

In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

SEED = 42
# Set Python random seed
import random
random.seed(SEED)
# Set NumPy random seed
np.random.seed(SEED)

# Set model parameters
hist_step = 60  # Number of history time steps
pred_step = 1   # Number of prediction time steps
input_channel = 3  # Number of input features: [downstream water level boundary, discharge flow 1, discharge flow 2]
output_channel = 10 # Number of output features
input_data_num = hist_step * input_channel  # 60 * 3 = 180

PyTorch version: 2.7.1+cu128
CUDA available: True
GPU device: NVIDIA GeForce RTX 5070 Ti
Using device: cuda:0


In [3]:
def median_absolute_error(y_true, y_pred):
    """Median Absolute Error (MdAE)"""
    errors = np.abs(y_true - y_pred)
    return np.median(errors)

def iqr_based_mae(y_true, y_pred):
    """
    IQR-based MAE: Calculate the MAE for samples where absolute errors fall within the [25%, 75%] range
    """
    errors = np.abs(y_true - y_pred).flatten()
    q1 = np.percentile(errors, 25)
    q3 = np.percentile(errors, 75)
    mask = (errors >= q1) & (errors <= q3)
    filtered_errors = errors[mask]
    if len(filtered_errors) == 0:
        return np.nan
    return np.mean(filtered_errors)

def hit_rate(y_true, y_pred, delta):
    """
    Hit rate calculation based on the maximum true value
    y_true, y_pred: [T, num_features]
    delta: Tolerance threshold (percentage, e.g., 5 means 5%)
    Returns hit rate (%)
    """
    max_val = np.max(np.abs(y_true))
    absolute_errors = np.abs(y_pred - y_true)
    hits = absolute_errors <= (delta / 100.0) * max_val
    return np.mean(hits) * 100

def calculate_all_metrics_horizontal_vertical(y_true, y_pred):
    """
    Calculate all metrics separately for horizontal and vertical velocities
    Assumes output order: horizontal1, vertical1, horizontal2, vertical2, ..., horizontal5, vertical5
    Returns dictionaries of metrics for horizontal and vertical components
    """
    # Extract horizontal velocities (values at positions 1,3,5,7,9)
    horizontal_true = y_true[:, [0, 2, 4, 6, 8]]
    horizontal_pred = y_pred[:, [0, 2, 4, 6, 8]]
    
    # Extract vertical velocities (values at positions 2,4,6,8,10)
    vertical_true = y_true[:, [1, 3, 5, 7, 9]]
    vertical_pred = y_pred[:, [1, 3, 5, 7, 9]]
    
    # Calculate horizontal metrics
    horizontal_metrics = {}
    horizontal_metrics['MSE'] = mean_squared_error(horizontal_true, horizontal_pred)
    horizontal_metrics['RMSE'] = np.sqrt(horizontal_metrics['MSE'])
    horizontal_metrics['MAE'] = mean_absolute_error(horizontal_true, horizontal_pred)
    horizontal_metrics['MdAE'] = median_absolute_error(horizontal_true, horizontal_pred)
    horizontal_metrics['IQR_MAE'] = iqr_based_mae(horizontal_true, horizontal_pred)
    
    # Calculate vertical metrics
    vertical_metrics = {}
    vertical_metrics['MSE'] = mean_squared_error(vertical_true, vertical_pred)
    vertical_metrics['RMSE'] = np.sqrt(vertical_metrics['MSE'])
    vertical_metrics['MAE'] = mean_absolute_error(vertical_true, vertical_pred)
    vertical_metrics['MdAE'] = median_absolute_error(vertical_true, vertical_pred)
    vertical_metrics['IQR_MAE'] = iqr_based_mae(vertical_true, vertical_pred)
    
    # Calculate hit rates (0-20%, step 1%)
    horizontal_hit_rates = {}
    vertical_hit_rates = {}
    
    for delta in range(0, 21):  # 0% to 20%
        horizontal_hit_rates[f'hit_rate_{delta}%'] = hit_rate(horizontal_true, horizontal_pred, delta)
        vertical_hit_rates[f'hit_rate_{delta}%'] = hit_rate(vertical_true, vertical_pred, delta)
    
    # Merge hit rates into respective metrics dictionaries
    horizontal_metrics.update(horizontal_hit_rates)
    vertical_metrics.update(vertical_hit_rates)
    
    return horizontal_metrics, vertical_metrics

def print_metrics(metrics_dict, label="Metrics"):
    """Print metrics dictionary"""
    print(f"\n{label}:")
    for key, value in metrics_dict.items():
        if 'hit_rate' in key:
            print(f"  {key}: {value:.2f}%")
        elif key in ['MSE', 'RMSE', 'MAE', 'MdAE', 'IQR_MAE']:
            print(f"  {key}: {value:.6f}")

In [4]:
def read_excel_file(filepath):
    """
    Read an Excel file, skipping the first row and the first column (ID numbers).
    Returns: numpy array with shape (number of samples, 190)
    """
    df = pd.read_excel(filepath, header=None)  # Do not automatically recognize headers
    # Skip the first row and the first column (ID numbers)
    values = df.iloc[1:, 1:].values  # Start from the second row and second column
    # Ensure all data are of float32 type
    values = values.astype('float32')
    print(f"File {os.path.basename(filepath)} shape: {values.shape}")
    return values

def prepare_test_data(values, scaler_input, scaler_output):
    """
    Prepare test data: split input and output, and perform normalization.
    values: raw data with shape (number of samples, 190)
    Returns: normalized input and output
    """
    # Split input and output
    x = values[:, :input_data_num].reshape(-1, hist_step, input_channel)
    y = values[:, input_data_num:]

    # Normalize input data
    x_reshaped = x.reshape(-1, input_channel)
    x_norm = scaler_input.transform(x_reshaped)
    x_norm = x_norm.reshape(-1, hist_step, input_channel)

    # Normalize output data
    y_norm = scaler_output.transform(y)

    return x_norm, y_norm, y  # Return normalized input, normalized output, and raw output

In [5]:
class Config:
    def __init__(self):
        self.seq_len = 60
        self.pred_len = 1
        self.d_model = 256
        self.d_state = 16
        self.d_ff = 256
        self.e_layers = 2
        self.d_layers = 1
        self.dropout = 0.32
        self.embed = 'timeF'
        self.freq = 'h'
        self.activation = 'gelu'
        self.output_attention = False
        self.use_norm = False
        self.class_strategy = 'cls'
        self.channel_independence = False
        self.enc_in = 3  # Number of input features
        self.dec_in = 3
        self.c_out = 10  # Number of output features
        self.factor = 1
        self.n_heads = 8
        self.distil = True
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create model configuration
args = Config()

def load_model(model_path, device='cpu'):
    """
    Load the complete model (including structure and parameters)
    """
    # Load the saved model
    checkpoint = torch.load(model_path, map_location=device, weights_only = False)
    
    # Get the model object
    model = checkpoint['model']
    
    # Get configuration (optional)
    config = checkpoint['config']
    
    # Set the model to evaluation mode
    model.eval()
    
    print(f"Complete model loaded from {model_path}")
    print(f"Model configuration: {config}")
    
    return model, config

# Load model weights
model_path = os.path.join('..', 'Model', 'Best_Flowformer.pth')
if os.path.exists(model_path):
    # Load the complete model (no need to import the Flowformer class)
    model, _ = load_model(model_path, device)
    print(f"Model successfully loaded and ready for evaluation mode")
else:
    print(f"Complete model file does not exist: {model_path}")

# Load scalers
scaler_path = os.path.join('..', 'Scaler')
scaler_input_path = os.path.join(scaler_path, 'scaler_input.pkl')
scaler_output_path = os.path.join(scaler_path, 'scaler_output.pkl')

if os.path.exists(scaler_input_path) and os.path.exists(scaler_output_path):
    scaler_input = joblib.load(scaler_input_path)
    scaler_output = joblib.load(scaler_output_path)
    print(f"Scalers loaded from {scaler_path}")
    print(f"Input scaler parameters: min={scaler_input.data_min_}, max={scaler_input.data_max_}")
    print(f"Output scaler parameters: min={scaler_output.data_min_}, max={scaler_output.data_max_}")
else:
    print(f"Error: Could not find scaler files")
    print(f"Please check if the following files exist:")
    print(f"  {scaler_input_path}")
    print(f"  {scaler_output_path}")
    # If files don't exist, try loading from a merged file
    scalers_path = os.path.join(scaler_path, 'scalers.pkl')
    if os.path.exists(scalers_path):
        print(f"Attempting to load from merged file: {scalers_path}")
        scalers = joblib.load(scalers_path)
        scaler_input = scalers['scaler_input']
        scaler_output = scalers['scaler_output']
        print("Scalers loaded from merged file")
    else:
        print("Unable to load scalers, testing cannot proceed")

Complete model loaded from ..\Model\Best_Flowformer.pth
Model configuration: {'seq_len': 60, 'pred_len': 1, 'd_model': 256, 'd_state': 16, 'd_ff': 256, 'e_layers': 2, 'd_layers': 1, 'dropout': 0.32, 'embed': 'timeF', 'freq': 'h', 'activation': 'gelu', 'output_attention': False, 'use_norm': False, 'class_strategy': 'cls', 'channel_independence': False, 'enc_in': 3, 'dec_in': 3, 'c_out': 10, 'factor': 1, 'n_heads': 8, 'distil': True}
Model successfully loaded and ready for evaluation mode
Scalers loaded from ..\Scaler
Input scaler parameters: min=[39.82  0.    0.  ], max=[ 45.27    625.5512  243.04709]
Output scaler parameters: min=[-0.07950529 -0.4396197  -0.08052862 -0.3627781  -0.09965846 -0.508818
 -0.14136367 -0.59327614 -0.17341709 -0.6513255 ], max=[0.01579557 0.01589902 0.016485   0.02688956 0.01606339 0.01044686
 0.01972071 0.01044966 0.04103037 0.01032581]


In [6]:
def predict_batch(model, x_norm, device='cpu'):
    """
    Make predictions for a batch of samples
    x_norm: Normalized input data with shape (batch_size, seq_len, input_channel)
    Returns: Normalized prediction output with shape (batch_size, output_channel)
    """
    model.eval()
    model.to(device)
    
    # Convert input data to tensor
    x_tensor = torch.tensor(x_norm, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        B, L, D = x_tensor.shape
        # Create time stamps (consistent with training)
        x_mark_enc = torch.zeros(B, L, 4).to(device)
        x_dec = torch.zeros(B, args.pred_len, D).to(device)
        x_mark_dec = torch.zeros(B, args.pred_len, 4).to(device)
        
        # Forward propagation
        out = model(x_tensor, x_mark_enc, x_dec, x_mark_dec)
        out = out[:, -1, :]  # [B, 10]
        
        # Convert to numpy
        output_norm = out.cpu().numpy()
    
    return output_norm

In [7]:
def test_single_condition(model, filepath, condition_name, scaler_input, scaler_output, device='cpu'):
    """
    Test a single condition
    """
    print(f"\n{'='*60}")
    print(f"Testing condition: {condition_name}")
    print(f"File: {os.path.basename(filepath)}")
    
    # 1. Read data
    values = read_excel_file(filepath)
    
    # 2. Prepare test data
    x_norm, y_norm, y_original = prepare_test_data(values, scaler_input, scaler_output)
    
    # 3. Make predictions
    y_pred_norm = predict_batch(model, x_norm, device)
    
    # 4. Inverse transform to get predictions in original scale
    y_pred_original = scaler_output.inverse_transform(y_pred_norm)
    
    # 5. Calculate metrics
    horizontal_metrics, vertical_metrics = calculate_all_metrics_horizontal_vertical(
        y_original, y_pred_original
    )
    
    # 6. Print results
    print_metrics(horizontal_metrics, "Horizontal Velocity Metrics")
    print_metrics(vertical_metrics, "Vertical Velocity Metrics")
    
    return {
        'condition_name': condition_name,
        'horizontal_metrics': horizontal_metrics,
        'vertical_metrics': vertical_metrics,
        'y_true': y_original,
        'y_pred': y_pred_original
    }

In [7]:
# Set test file paths
test_data_dir = os.path.join('..', 'TestData')
test_files = [
    ("Condition 1", os.path.join(test_data_dir, "Condition_1_32_sametime.xlsx")),
    ("Condition 9", os.path.join(test_data_dir, "Condition_9_23_600.xlsx")),
    ("Condition 12", os.path.join(test_data_dir, "Condition_12_32_750.xlsx")),
    ("Condition 15", os.path.join(test_data_dir, "Condition_15_23_1050.xlsx")),
    ("Condition 22", os.path.join(test_data_dir, "Condition_22_32_1500.xlsx"))
]

# Check if files exist
print("Checking if test files exist:")
all_files_exist = True
for condition_name, filepath in test_files:
    if os.path.exists(filepath):
        print(f"  ✓ {condition_name}: {os.path.basename(filepath)}")
    else:
        print(f"  ✗ {condition_name}: File does not exist - {filepath}")
        all_files_exist = False

if not all_files_exist:
    print("Error: Some test files are missing, please check the file paths")
else:
    print("\nAll test files exist, starting testing...")

Checking if test files exist:
  ✓ Condition 1: Condition_1_32_sametime.xlsx
  ✓ Condition 9: Condition_9_23_600.xlsx
  ✓ Condition 12: Condition_12_32_750.xlsx
  ✓ Condition 15: Condition_15_23_1050.xlsx
  ✓ Condition 22: Condition_22_32_1500.xlsx

All test files exist, starting testing...


In [8]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def calculate_max_error(y_true, y_pred):
    """Calculate maximum error"""
    return np.max(np.abs(y_true - y_pred))

def calculate_all_metrics_horizontal_vertical(y_true, y_pred):
    """
    Calculate all metrics separately for horizontal and vertical velocities
    Assumes output order: horizontal1, vertical1, horizontal2, vertical2, ..., horizontal5, vertical5
    Returns dictionaries of metrics for horizontal and vertical components
    """
    # Extract horizontal velocities (values at positions 1,3,5,7,9)
    horizontal_true = y_true[:, [0, 2, 4, 6, 8]]
    horizontal_pred = y_pred[:, [0, 2, 4, 6, 8]]
    
    # Extract vertical velocities (values at positions 2,4,6,8,10)
    vertical_true = y_true[:, [1, 3, 5, 7, 9]]
    vertical_pred = y_pred[:, [1, 3, 5, 7, 9]]
    
    # Calculate horizontal metrics
    horizontal_metrics = {}
    horizontal_metrics['MSE'] = mean_squared_error(horizontal_true, horizontal_pred)
    horizontal_metrics['RMSE'] = np.sqrt(horizontal_metrics['MSE'])
    horizontal_metrics['MAE'] = mean_absolute_error(horizontal_true, horizontal_pred)
    horizontal_metrics['MdAE'] = median_absolute_error(horizontal_true, horizontal_pred)
    horizontal_metrics['IQR_MAE'] = iqr_based_mae(horizontal_true, horizontal_pred)
    horizontal_metrics['ME'] = calculate_max_error(horizontal_true, horizontal_pred)  # Added maximum error
    horizontal_metrics['R2'] = r2_score(horizontal_true, horizontal_pred)  
    horizontal_metrics['CA'] = 0.33*(horizontal_metrics['RMSE'] + horizontal_metrics['MAE'] + (1-horizontal_metrics['R2'])) 
    
    # Calculate vertical metrics
    vertical_metrics = {}
    vertical_metrics['MSE'] = mean_squared_error(vertical_true, vertical_pred)
    vertical_metrics['RMSE'] = np.sqrt(vertical_metrics['MSE'])
    vertical_metrics['MAE'] = mean_absolute_error(vertical_true, vertical_pred)
    vertical_metrics['MdAE'] = median_absolute_error(vertical_true, vertical_pred)
    vertical_metrics['IQR_MAE'] = iqr_based_mae(vertical_true, vertical_pred)
    vertical_metrics['ME'] = calculate_max_error(vertical_true, vertical_pred)  # Added maximum error
    vertical_metrics['R2'] = r2_score(vertical_true, vertical_pred)  
    vertical_metrics['CA'] = 0.33*(vertical_metrics['RMSE'] + vertical_metrics['MAE'] + (1-vertical_metrics['R2']))
    
    # Calculate hit rates (0-20%, step 1%)
    horizontal_hit_rates = {}
    vertical_hit_rates = {}
    
    for delta in range(0, 21):  # 0% to 20%
        horizontal_hit_rates[f'hit_rate_{delta}%'] = hit_rate(horizontal_true, horizontal_pred, delta)
        vertical_hit_rates[f'hit_rate_{delta}%'] = hit_rate(vertical_true, vertical_pred, delta)
    
    # Merge hit rates into respective metrics dictionaries
    horizontal_metrics.update(horizontal_hit_rates)
    vertical_metrics.update(vertical_hit_rates)
    
    return horizontal_metrics, vertical_metrics

def print_metrics(metrics_dict, label="Metrics"):
    """Print metrics dictionary"""
    print(f"\n{label}:")
    # Print main metrics first
    for key in ['MSE', 'RMSE', 'MAE', 'MdAE', 'IQR_MAE', 'ME', 'R2', 'CA']:
        if key in metrics_dict:
            print(f"  {key}: {metrics_dict[key]:.6f}")
    print(" ")
    # Then print hit rates
    for key, value in metrics_dict.items():
        if 'hit_rate' in key:
            print(f"  {key}: {value:.2f}%")

def save_flow_predictions_to_excel(all_results, save_path='Flowformer_Actual_Predicted.xlsx'):
    """
    Save actual and predicted flow velocity values for all conditions to different sheets in one Excel file
    """
    # Create Excel writer
    with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
        # Process each condition
        for result in all_results:
            condition_name = result['condition_name']
            y_true = result['y_true']
            y_pred = result['y_pred']
            
            # Create column names
            column_names = []
            for i in range(1, 6):  # 1 to 5, 5 points total
                column_names.extend([
                    f'U_Actual_{i}',
                    f'V_Actual_{i}',
                    f'U_Predicted_{i}',
                    f'V_Predicted_{i}'
                ])
            
            # Extract data
            # Actual values: horizontal velocities (positions 1,3,5,7,9)
            u_true = y_true[:, [0, 2, 4, 6, 8]]  # [n_samples, 5]
            v_true = y_true[:, [1, 3, 5, 7, 9]]  # [n_samples, 5]
            
            # Predicted values: horizontal velocities (positions 1,3,5,7,9)
            u_pred = y_pred[:, [0, 2, 4, 6, 8]]  # [n_samples, 5]
            v_pred = y_pred[:, [1, 3, 5, 7, 9]]  # [n_samples, 5]
            
            # Create data array
            data_list = []
            for i in range(5):  # 5 points
                data_list.append(u_true[:, i])  # U_Actual_i
                data_list.append(v_true[:, i])  # V_Actual_i
                data_list.append(u_pred[:, i])  # U_Predicted_i
                data_list.append(v_pred[:, i])  # V_Predicted_i
            
            # Combine data
            data = np.column_stack(data_list)
            
            # Create DataFrame
            df = pd.DataFrame(data, columns=column_names)
            
            # Write DataFrame to Excel as a separate sheet, sheet name is condition name
            sheet_name = f"{condition_name.replace('Condition', '').strip()}"
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            print(f"  {condition_name} data saved to sheet: {sheet_name}")
    
    print(f"\nAll condition data saved to: {save_path}")
    return save_path

def test_single_condition(model, filepath, condition_name, scaler_input, scaler_output, device='cpu'):
    """
    Test a single condition
    """
    print(f"\n{'='*60}")
    print(f"Testing condition: {condition_name}")
    print(f"File: {os.path.basename(filepath)}")
    
    # 1. Read data
    values = read_excel_file(filepath)
    
    # 2. Prepare test data
    x_norm, y_norm, y_original = prepare_test_data(values, scaler_input, scaler_output)
    
    # 3. Make predictions
    y_pred_norm = predict_batch(model, x_norm, device)
    
    # 4. Inverse transform to get predictions in original scale
    y_pred_original = scaler_output.inverse_transform(y_pred_norm)
    
    # 5. Calculate metrics
    horizontal_metrics, vertical_metrics = calculate_all_metrics_horizontal_vertical(
        y_original, y_pred_original
    )
    
    # 6. Print results
    print_metrics(horizontal_metrics, "Horizontal Velocity Metrics")
    print_metrics(vertical_metrics, "Vertical Velocity Metrics")
    
    return {
        'condition_name': condition_name,
        'horizontal_metrics': horizontal_metrics,
        'vertical_metrics': vertical_metrics,
        'y_true': y_original,
        'y_pred': y_pred_original
    }

def test_all_conditions(model, test_files, scaler_input, scaler_output, device='cpu'):
    """
    Test all conditions
    """
    all_results = []
    all_y_true = []
    all_y_pred = []
    
    print("Starting to test all conditions...")
    
    for condition_name, filepath in test_files:
        result = test_single_condition(model, filepath, condition_name, scaler_input, scaler_output, device)
        all_results.append(result)
        all_y_true.append(result['y_true'])
        all_y_pred.append(result['y_pred'])
    
    # Combine data from all conditions
    combined_y_true = np.vstack(all_y_true)
    combined_y_pred = np.vstack(all_y_pred)
    
    # Save all condition data to different sheets in one Excel file
    excel_path = save_flow_predictions_to_excel(all_results, 'Flowformer_Actual_Predicted.xlsx')
    
    # Calculate overall metrics
    print(f"\n{'='*60}")
    print("Overall test results (all conditions combined)")
    print(f"Total samples: {combined_y_true.shape[0]}")
    
    horizontal_metrics, vertical_metrics = calculate_all_metrics_horizontal_vertical(
        combined_y_true, combined_y_pred
    )
    
    print_metrics(horizontal_metrics, "Overall Horizontal Velocity Metrics")
    print_metrics(vertical_metrics, "Overall Vertical Velocity Metrics")
    
    return all_results, horizontal_metrics, vertical_metrics, excel_path

# Execute testing
if all_files_exist:
    all_results, horizontal_metrics, vertical_metrics, excel_path = test_all_conditions(
        model, test_files, scaler_input, scaler_output, device
    )
    
    # Read and display first few rows of each sheet
    print(f"\nExcel file {excel_path} content preview:")
    with pd.ExcelFile(excel_path) as xls:
        for sheet_name in xls.sheet_names:
            print(f"\nSheet: {sheet_name}")
            df = pd.read_excel(xls, sheet_name=sheet_name)
            print(f"Data shape: {df.shape}")
            print(f"First 3 rows:")
            print(df.head(3))
    
    print("\nTesting completed!")
else:
    print("\nTesting stopped due to missing test files")

Starting to test all conditions...

Testing condition: Condition 1
File: Condition_1_32_sametime.xlsx
File Condition_1_32_sametime.xlsx shape: (836, 190)

Horizontal Velocity Metrics:
  MSE: 0.000031
  RMSE: 0.005542
  MAE: 0.004354
  MdAE: 0.003638
  IQR_MAE: 0.003688
  ME: 0.020628
  R2: 0.917385
  CA: 0.030528
 
  hit_rate_0%: 0.00%
  hit_rate_1%: 11.53%
  hit_rate_2%: 24.93%
  hit_rate_3%: 37.22%
  hit_rate_4%: 46.65%
  hit_rate_5%: 56.67%
  hit_rate_6%: 65.72%
  hit_rate_7%: 72.63%
  hit_rate_8%: 78.54%
  hit_rate_9%: 83.37%
  hit_rate_10%: 87.51%
  hit_rate_11%: 90.57%
  hit_rate_12%: 93.30%
  hit_rate_13%: 95.14%
  hit_rate_14%: 96.58%
  hit_rate_15%: 97.39%
  hit_rate_16%: 97.89%
  hit_rate_17%: 98.56%
  hit_rate_18%: 98.88%
  hit_rate_19%: 99.16%
  hit_rate_20%: 99.28%

Vertical Velocity Metrics:
  MSE: 0.001017
  RMSE: 0.031897
  MAE: 0.023803
  MdAE: 0.018481
  IQR_MAE: 0.018983
  ME: 0.125778
  R2: 0.687610
  CA: 0.121470
 
  hit_rate_0%: 0.00%
  hit_rate_1%: 9.38%
  hit_ra

In [11]:
# Save test results to an Excel file
def save_test_results(all_results, horizontal_metrics, vertical_metrics, save_path):
    """Save test results to an Excel file"""
    import pandas as pd
    
    # Create a DataFrame to save the results
    results_data = []
    
    # Add results for each condition
    for result in all_results:
        condition_name = result['condition_name']
        h_metrics = result['horizontal_metrics']
        v_metrics = result['vertical_metrics']
        
        # Add horizontal metrics
        row = {'Condition': condition_name, 'Velocity_Type': 'Horizontal'}
        for key in ['MSE', 'RMSE', 'MAE', 'MdAE', 'IQR_MAE', 'ME']:
            row[key] = h_metrics[key]
        for delta in range(0, 21):
            row[f'hit_rate_{delta}%'] = h_metrics[f'hit_rate_{delta}%']
        results_data.append(row)
        
        # Add vertical metrics
        row = {'Condition': condition_name, 'Velocity_Type': 'Vertical'}
        for key in ['MSE', 'RMSE', 'MAE', 'MdAE', 'IQR_MAE', 'ME']:
            row[key] = v_metrics[key]
        for delta in range(0, 21):
            row[f'hit_rate_{delta}%'] = v_metrics[f'hit_rate_{delta}%']
        results_data.append(row)
    
    # Add overall results
    row = {'Condition': 'Overall', 'Velocity_Type': 'Horizontal'}
    for key in ['MSE', 'RMSE', 'MAE', 'MdAE', 'IQR_MAE', 'ME']:
        row[key] = horizontal_metrics[key]
    for delta in range(0, 21):
        row[f'hit_rate_{delta}%'] = horizontal_metrics[f'hit_rate_{delta}%']
    results_data.append(row)
    
    row = {'Condition': 'Overall', 'Velocity_Type': 'Vertical'}
    for key in ['MSE', 'RMSE', 'MAE', 'MdAE', 'IQR_MAE', 'ME']:
        row[key] = vertical_metrics[key]
    for delta in range(0, 21):
        row[f'hit_rate_{delta}%'] = vertical_metrics[f'hit_rate_{delta}%']
    results_data.append(row)
    
    # Create DataFrame and save
    df = pd.DataFrame(results_data)
    df.to_csv(save_path, index=False)
    print(f"Test results saved to: {save_path}")
    
    # Also save to Excel file
    excel_path = save_path  
    df.to_excel(excel_path, index=False, sheet_name='Test_Results')
    print(f"Test results saved to Excel: {excel_path}")
    
    return df, excel_path

# Save results
if all_files_exist:
    results_save_path = os.path.join('.', 'Test Metric.xlsx')
    results_df, results_excel_path = save_test_results(all_results, horizontal_metrics, vertical_metrics, results_save_path)
    
    # Display first few rows
    print("\nTest results preview:")
    print(results_df.head())

Test results saved to: .\Test Metric.xlsx
Test results saved to Excel: .\Test Metric.xlsx

Test results preview:
      Condition Velocity_Type       MSE      RMSE       MAE      MdAE  \
0   Condition 1    Horizontal  0.000031  0.005542  0.004354  0.003638   
1   Condition 1      Vertical  0.001017  0.031897  0.023803  0.018481   
2   Condition 9    Horizontal  0.000105  0.010264  0.007735  0.005501   
3   Condition 9      Vertical  0.001458  0.038181  0.029342  0.020073   
4  Condition 12    Horizontal  0.000052  0.007245  0.006089  0.005355   

    IQR_MAE        ME  hit_rate_0%  hit_rate_1%  ...  hit_rate_11%  \
0  0.003688  0.020628          0.0    11.531100  ...     90.574163   
1  0.018983  0.125778          0.0     9.377990  ...     72.464115   
2  0.005887  0.028956          0.0     4.621212  ...     47.632576   
3  0.023766  0.093079          0.0     4.450758  ...     49.128788   
4  0.005485  0.019310          0.0     3.291442  ...     52.866547   

   hit_rate_12%  hit_rate_1